In [1]:
import os
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

In [2]:
data_path = Path(os.environ["DATA_PATH"])
initial_path = data_path / "initial"
generated_path = data_path / "generated"

population_grids_path = Path(os.environ["POPULATION_GRIDS_PATH"])

# Geometría

In [ ]:
df_geom = (
    gpd.read_file(initial_path / "r_02a_zonas", columns=["CVEGEO", "_zonaurban"])
    .rename(columns={"_zonaurban": "zone"})
    .query("zone.notna()")
    .assign(zone=lambda df: df["zone"].astype(int).sub(1).astype("category"))
    .set_index("CVEGEO")
    .to_crs("EPSG:6372")
)

# Ingreso

In [18]:
income = gpd.read_file(initial_path / "income.gpkg").set_index("cvegeo")["income_pc"]

# Área baldía

In [25]:
df_barren = (
    gpd.read_file(initial_path / "baldios")
    .assign(geometry=lambda df: df["geometry"].force_2d())
    .to_crs("EPSG:6372")
)
barren_area_frac = (
    df_geom.reset_index()
    .overlay(df_barren, how="intersection")
    .assign(area=lambda df: df["geometry"].area)
    .groupby("CVEGEO")["area"]
    .sum()
    .div(df_geom["geometry"].area)
)

# Delitos

In [33]:
crimes = gpd.read_file(initial_path / "num_delitos_ageb").set_index("CVEGEO")[
    "num_delito"
]

# Valor catastral

In [51]:
df_cat = (
    gpd.read_file(
        initial_path / "valor_cat_actualizado",
        columns=["secsub", "valor2024"],
    )
    .dropna(subset=["secsub"])
    .drop(columns=["secsub"])
    .assign(geometry=lambda df: df["geometry"].force_2d())
    .to_crs("EPSG:6372")
)

# Censo

In [ ]:
wanted_cols = [
    "POBTOT",
    "P_60YMAS",
    "P_0A2",
    "P_3A5",
    "P_6A11",
    "P_12A14",
    "P_15A17",
    "P_18A24",
    "POB15_64",
    "TVIVPAR",
    "TVIVPARHAB",
    "GRAPROES",
]

df_census = (
    pd.read_csv(
        initial_path / "conjunto_de_datos_ageb_urbana_02_cpv2020.csv",
        usecols=[*wanted_cols, "ENTIDAD", "MUN", "LOC", "AGEB", "MZA"],
    )
    .query("MZA == 0")
    .assign(
        CVEGEO=lambda df: df["ENTIDAD"].astype(str).str.zfill(2)
        + df["MUN"].astype(str).str.zfill(3)
        + df["LOC"].astype(str).str.zfill(4)
        + df["AGEB"].astype(str).str.zfill(4),
    )
    .drop(
        columns=[
            "ENTIDAD",
            "MUN",
            "LOC",
            "AGEB",
            "MZA",
        ],
    )
    .set_index("CVEGEO")[wanted_cols]
    .replace("*", np.nan)
)

for c in df_census.columns:
    if c.startswith("P") and df_census[c].dtype == "str":
        try:
            df_census[c] = df_census[c].astype(int)
        except ValueError:
            df_census[c] = df_census[c].astype(float)

# Final

In [37]:
df_agebs = (
    df_geom.join(df_census, how="inner")
    .assign(
        income=income,
        vacant_area_frac=barren_area_frac,
        crimes=crimes,
        area_km2=lambda df: df["geometry"].area.div(1e6),
    )
    .sort_index()
)
df_agebs.to_file(generated_path / "agebs.gpkg")

In [38]:
df_agebs.head(5)

,zone,geometry,POBTOT,P_60YMAS,P_0A2,P_3A5,P_6A11,P_12A14,P_15A17,P_18A24,POB15_64,TVIVPAR,TVIVPARHAB,GRAPROES,income,vacant_area_frac,crimes,area_km2
CVEGEO,,,,,,,,,,,,,,,,,,
0200200010019,5,"POLYGON ((1240836.538 2335789.786, 1240836.059...",2943,91.0,139.0,176.0,431.0,169.0,143.0,325.0,1974.0,1032,889,9.08,4.769395,0.128376,278.0,0.297102
0200200010023,5,"POLYGON ((1240864.547 2335052.842, 1240781.408...",2377,71.0,120.0,138.0,314.0,142.0,115.0,270.0,1627.0,960,765,9.31,4.931883,0.070573,287.0,0.264551
0200200010038,5,"POLYGON ((1239641.963 2334838.629, 1239639.679...",6104,100.0,419.0,473.0,750.0,288.0,256.0,911.0,4125.0,2458,1969,9.23,4.373754,0.068269,245.0,0.564767
0200200010042,3,"POLYGON ((1225449.578 2348799.553, 1225449.093...",1293,294.0,20.0,42.0,81.0,35.0,53.0,124.0,903.0,520,447,9.42,5.048268,0.049958,234.0,0.453839
0200200010057,3,"POLYGON ((1225449.578 2348799.553, 1225744.357...",1465,297.0,26.0,30.0,99.0,45.0,50.0,142.0,1072.0,698,605,8.77,4.788611,0.065495,529.0,0.445666
